# 🧠 LSTM — Training Notes

## Backpropagation Through Time (BPTT) & Gradient Flow

![Image](https://images.openai.com/static-rsc-4/xIp-M9cIy0TneVO4UcGkzP6hKYoMlOAn8p-R-In9EqRxxPdqGcRtwwfq38LRC9qnFqElA-EQ2JKx7avnswtxfgxmhm8uDHCIe3NK_Ueyqb6ZiEmO4FqbcTGd4t3dLsreeJBcH1gODooZpWkM-WbxU8P7OEYHxFDWUcwt_XRKfSrr7L7uP5Heul5rx8PfoRg4?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/fXpr5KpXoCqsLHgk7VbR4UaA6q8sI_U1rbegPAfgeP_uT_fuUdTfuDS310cw2ZzOIhBy29PoqeKfeZickQxlaHyOJE3f4J7r8KJQ6BqxS8h141O5NET3I2brZAnq4c2KhB2Aj_LRc5LNPqHENrxdXEsVq-nYe5xCDqIlEo4h09HrWCj1xKSDze0Zb4Oh4wVZ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/-6QwhF5awc5IG10vYzl-2W89d5m6-1jBdJnpbqwjwpkeMRqEV-lEcEkPhKKRzasdsZTZV2GGypB8M5ogsvRTMwMJ0Eq2xDvaM-imglmKLrZQF9rvToQEwFFFr5GSB8HB5WM0OVMFzOQNvuBfp_DvRfDWsZrwouNrVb7bKw0UvfHPM3Xf-2KdkcqnXmz0mJdX?purpose=fullsize)

## 1. What is BPTT?

**BPTT = Backpropagation Through Time.**

An LSTM processes sequential data one time step at a time:

```text
x₁ → LSTM → h₁
          ↓
x₂ → LSTM → h₂
          ↓
x₃ → LSTM → h₃
          ↓
          ...
xₜ → LSTM → hₜ
```

During training, we **unroll** the LSTM across time and apply backpropagation through these time steps.

So instead of thinking of the LSTM as one isolated cell:

```text
       ┌───────┐
xₜ ───►│ LSTM  │───► hₜ
       └───────┘
```

we conceptually view it as:

```text
x₁ ─► LSTM ─► h₁ ─►
                  │
x₂ ─► LSTM ─► h₂ ─►
                  │
x₃ ─► LSTM ─► h₃ ─►
                  │
                 ...
                  │
xₜ ─► LSTM ─► hₜ ─► Loss
```

The **same LSTM parameters are reused at every time step**.

---

# 2. Forward Pass

During the forward pass, information moves **forward through time**.

```text
x₁ → h₁ → h₂ → h₃ → ... → hₜ → Prediction → Loss
      ↑    ↑    ↑             ↑
     C₁   C₂   C₃            Cₜ
```

At each time step:

$$
(x_t,h_{t-1},C_{t-1})
\rightarrow
(h_t,C_t)
$$

The LSTM uses:

* current input \(x_t\)
* previous hidden state \(h_{t-1}\)
* previous cell state \(C_{t-1}\)

to calculate:

* current hidden state \(h_t\)
* current cell state \(C_t\)

---

# 3. Backward Pass Through Time

After calculating the loss, the network calculates gradients.

The error travels **backward through the sequence**:

```text
Forward:

x₁ → x₂ → x₃ → x₄ → x₅ → Loss
                              ↓
Backward:

x₁ ← x₂ ← x₃ ← x₄ ← x₅ ← Loss
```

More precisely:

```text
Loss
 ↓
h₅
 ↓
h₄
 ↓
h₃
 ↓
h₂
 ↓
h₁
```

This is why it is called:

> **Backpropagation Through Time**

because the recurrent network is unrolled across time and gradients are propagated through those time steps.

---

# 4. Gradient Flow Through Time

The important question is:

> **How does the gradient travel from later time steps back to earlier time steps?**

Consider:

```text
C₁ → C₂ → C₃ → C₄ → C₅
                         ↓
                       Loss
```

During backpropagation:

```text
Loss
 ↓
C₅
 ↓
C₄
 ↓
C₃
 ↓
C₂
 ↓
C₁
```

The gradient must travel through many recurrent operations.

This creates the **gradient-flow problem**.

---

# 5. Why Vanilla RNNs Have a Problem

In a vanilla RNN:

$$
h_t=\tanh(W_{xh}x_t+W_{hh}h_{t-1}+b)
$$

The gradient repeatedly passes through the recurrent computation.

Conceptually:

$$
\frac{\partial L}{\partial h_1}
=
\frac{\partial L}{\partial h_T}
\frac{\partial h_T}{\partial h_{T-1}}
\cdots
\frac{\partial h_2}{\partial h_1}
$$

There may be **many multiplication operations**.

If these factors are generally smaller than 1:

```text
1.0 → 0.7 → 0.5 → 0.3 → 0.1 → 0.01 → ...
```

the gradient can become extremely small.

### This is the:

## 🔻 Vanishing Gradient Problem

The early time steps receive very little gradient information.

Therefore, the network struggles to learn relationships between events that are far apart in a sequence.

---

# 6. How LSTM Helps Gradient Flow

LSTM introduces a dedicated **cell state pathway**:

```text
C₁ ─────────► C₂ ─────────► C₃ ─────────► C₄
 │             │             │             │
 ▼             ▼             ▼             ▼
Gates         Gates         Gates         Gates
```

The key cell-state equation is:

$$
C_t=f_t\odot C_{t-1}
+i_t\odot\tilde C_t
$$

There is a direct recurrent path from:

$$
C_{t-1}\rightarrow C_t
$$

controlled primarily by the **forget gate**.

---

# 7. Role of the Forget Gate in Gradient Flow

The forget gate is:

$$
f_t=\sigma(W_f[h_{t-1},x_t]+b_f)
$$

Since sigmoid produces values between 0 and 1:

$$
0\leq f_t\leq1
$$

Suppose:

$$
f_t\approx1
$$

Then:

$$
C_t\approx C_{t-1}
$$

So information stored in the cell state can be preserved.

Conceptually:

```text
fₜ ≈ 0
Cₜ₋₁ ──X──► Cₜ
       forget

fₜ ≈ 1
Cₜ₋₁ ─────► Cₜ
       preserve
```

This also helps gradients propagate through the cell-state pathway.

---

# 8. Important Gradient Relationship

From:

$$
C_t=f_t\odot C_{t-1}
+i_t\odot\tilde C_t
$$

the direct contribution of the previous cell state is controlled by \(f_t\).

Conceptually:

$$
\frac{\partial C_t}{\partial C_{t-1}}
\approx f_t
$$

Therefore, when:

$$
f_t\approx1
$$

the cell can preserve information and allow a stronger gradient pathway across time.

### 🧠 Intuition

```text
Forget gate ≈ 1
       ↓
Keep old memory
       ↓
Cell state changes less
       ↓
Information can survive longer
       ↓
Gradient can propagate more effectively
```

---

# 9. Shared Parameters During BPTT

One very important concept:

The LSTM **does not have completely different weights for every time step**.

Instead:

```text
Time 1       Time 2       Time 3
 ┌─────┐      ┌─────┐      ┌─────┐
 │LSTM │      │LSTM │      │LSTM │
 └─────┘      └─────┘      └─────┘
    ↑            ↑            ↑
    └────────────┴────────────┘
          SAME PARAMETERS
```

For example, the same \(W_f\) is used at every time step.

Therefore, during BPTT, each time step contributes to the gradient of the **same parameters**.

Conceptually:

$$
\frac{\partial L}{\partial W}
=
\sum_t
\frac{\partial L}{\partial W}\bigg|_t
$$

So gradients from different time steps are accumulated to update the shared parameters.

---

# 10. Complete BPTT Flow

![Image](https://images.openai.com/static-rsc-4/0AdEk9uV6UaeOAzz6iHGn0B6knF2rhOZlkthfA5cqCU_pkXIW94WCYbdranNVLv4okbpxy7RUV6CA6oUq6Lmq-on49aKoBUfcRevtrs2QJh8yOPK43f0MQvjNvdnDn34kQsN8l07D-1WrQ9NBQGCupA8RzAanxJLPGPa0JHsYc0ZQvYOEjcy4ZUMvjXXCoEt?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/aa35J2im_uFUeMpudt91vrvJv4uCVWOPU83dDpVD6a20Q0uphuw_s-Rmo1tOl_C3xgGvBxN3ADRtAFCwJP50Qlp0Ha8tIy8_NRPoQezHwo5YEM4m15uF9GEg0bEYu19j_97y2GtVHX8M12GLfqOTl6_YRiz13zymbuaYbCfcYES_-6DTchD9f2zVSuj8oSCw?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/mbJWnOkJNkHI20I8H-DwXVYbpw-7Tm1XQ76Kr3tT8ri_5TrAbO6OvzEKtlfSpAkih2MCK-UYCq7pIOos_W1vyjd9okMTnqxfujCe9uKsDsJuZAVjndv1_kTgQ7uN7kREY-9TdwnMmLcxroo-biP2qcISH3tAvj3rz7_UhfnjkbZj3I7ckILqqVKHAEUTfkBw?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/IFCubwSKbUaMYiy-C9EcNDsyY8Nhme9MyeT0zDb1g27Kg36FDhqKFsqW2WU6MQfgKcknpgp7JpCm5dqFKa3VVz_i-FAoxtZkd_VX0WbGtOjVudaNhaj_r-0MM5u7JipcnSrDZWQOctHFGfeQ1wgD9iS4HqxdjSzTZthmTZ2FrnutcVe3PMlNtvdaCT7xs_g5?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/cLRcAAgYlI2hWAfWS6PUy5mpxvIgnDubRP-crCgz0vXVw9wSZaARUYn1ArOgOVwNoiVyN7EMpYIYWZVQJaWxPC1F8fyFzy5jcg98Cvb6ITzeNunUwOTsx_C2jIM-eUPmfi8L57X5b_QasXf6Uqe662FdBX0NaXtZED6k4E0NMoupJZZ4NwU6W6wS169Sm136?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/eW7IJpXjpBGH4268OGMUUSMS_HG3fQsdV1PGv4BFTUmu3c791gX0wfGki4bRj8lavUuZSErp2jTLFWmo0iQr0o7yIgl4tuch15AgYBIuT-MLbGh7Vr6UBPvD6B88MdFxPpXg53vwWEo_M-1YQ1zzkcZvFEaXw0xKtQSQqqhknBWTZFiSeEIewiCq4u-74Bh6?purpose=fullsize)

### Forward:

```text
x₁ → LSTM → x₂ → LSTM → x₃ → LSTM → ... → xₜ
                                      ↓
                                    Loss
```

### Backward:

```text
Loss
 ↓
Time t
 ↓
Time t-1
 ↓
Time t-2
 ↓
...
 ↓
Time 1
```

At every step, gradients flow through:

* hidden state
* cell state
* forget gate
* input gate
* candidate state
* output gate

and ultimately contribute to the gradients of the shared LSTM parameters.

---

# 11. Does LSTM Completely Solve Vanishing Gradients?

⚠️ **No.**

This is an important interview point.

LSTM does **not guarantee that vanishing gradients will never occur**.

Instead, LSTM introduces:

* cell state
* gates
* controlled information flow
* a more direct memory pathway

which can make long-term information and gradient propagation **much more robust than in a basic vanilla RNN**.

So remember:

> **LSTM mitigates the vanishing-gradient problem; it does not magically eliminate every possible gradient-flow problem.**

---

# 🧠 RNN vs LSTM — Gradient Flow

| Feature                      | Vanilla RNN                 | LSTM                |
| ---------------------------- | --------------------------- | ------------------- |
| Main recurrent memory        | Hidden state                | Cell + hidden state |
| Long-term dependency         | Difficult                   | Better supported    |
| Gradient flow                | Can degrade over many steps | More controlled     |
| Vanishing gradients          | Major problem               | Mitigated           |
| Gating mechanism             | ❌                           | ✅                   |
| Dedicated cell-state pathway | ❌                           | ✅                   |

---

# 🔥 Exam / Interview Memory Points

### BPTT

**Backpropagation Through Time** = backpropagation applied to an unrolled recurrent network.

### Forward direction

$$
x_1\rightarrow x_2\rightarrow\cdots\rightarrow x_T
$$

### Backward direction

$$
L\rightarrow T\rightarrow T-1\rightarrow\cdots\rightarrow1
$$

### LSTM memory equation

$$
\boxed{
C_t=f_t\odot C_{t-1}+i_t\odot\tilde C_t
}
$$

### Important relationship

$$
\boxed{
\frac{\partial C_t}{\partial C_{t-1}}\approx f_t
}
$$

Conceptually, \(f_t\approx1\) allows the cell state to preserve information and provides a stronger route for gradient propagation.

### Most important intuition

> **RNN repeatedly transforms its hidden state, which can make gradients vanish over long sequences. LSTM adds a gated cell-state pathway that gives information and gradients a more controlled route through time.**

![Image](https://images.openai.com/static-rsc-4/NPYT29M1saTvfFZjLTNaJzGcjXUu5fF9nWCZ7oQCKnj4zgNaY69isS2DdY1KjCI9wmYC8vNxOiPpyed7aEVAT25WSAoplWffgQurwKInx8t8RAFN20qcdMzmghgMCUGswQIvD7zVUtpeLy38zZcG4o-ljaMIRNaijZPRYJmwBc4Y_HV9wkTGoUWznkBC1EiZ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/0AdEk9uV6UaeOAzz6iHGn0B6knF2rhOZlkthfA5cqCU_pkXIW94WCYbdranNVLv4okbpxy7RUV6CA6oUq6Lmq-on49aKoBUfcRevtrs2QJh8yOPK43f0MQvjNvdnDn34kQsN8l07D-1WrQ9NBQGCupA8RzAanxJLPGPa0JHsYc0ZQvYOEjcy4ZUMvjXXCoEt?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/n3ntWGvxQ8qc5miLHNM14YA0ghUj2Z7j_LPQXEkVO6nImhjqiOVCD6FJL00u1Xy5uIlR3vAo2FFx9FVNZu1xjxVCDjcSZ4_DYOi41Z2BMIyNxBganQgux4Oog5VkOrcSheOSF9W4K-hCtSBjC7Kd-3uSVj9c58ZfiP0ItRMgRknt3FCM5rzPrI2tKb9K5DP-?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/7v11aepxIeO0o8zUcjmb5foDLeIaKpcgqa-Lj45GwOb-effd76vCGB2sMIZIm6i76u_Uagb8L9zpjl84BNsYUH7uHoIA88V9mWs36Csk5gIk59vjY-ZKvmKRdIxM8Ixrwky9lkqvIBl0eTvNXlz2QJNSVwWqP27k_F_B33T2Z6usv3bbiIdkGHfDguJyReYH?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/cNzKB61Oqnm2zosxwWq97onL-OMULwxhYg87kyevbGO6EnGp9gEB7Is2Fnl8wa-ObkNLsPZWYI3MC40IV36atqOqWE-TtXeNeXGs3QbdooHq_l3HUlGchWM22aMWck5swmAVsdVClX7ilg6VrQUtOOPx_KWwq9P6q93JSTn0PHSVDuLsOB2mUexUCbY13Sk0?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/e7FNDM_0xLLj3ss5oC6lxjc2hcOs6iI7ecCeMeVIBzCq8oJhFK56F0WOtNUiSOwN1xF21z5--SMwsc5AnjZG5geF3c4Tccs8qUEvF4cTtlQJj5xmaQfD5b2iZMuEqA7KhxQNpfi4qkuf2G09apR6uSQAKbUKhJhuKhAiGACWCQ4Wy1lEWpGAmd4xzAUPj-SF?purpose=fullsize)

---